# Week 70, Day 6: Exam debrief

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w070d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


In [ ]:
from collections import deque

def interleave(*iterables):
    active = deque(iter(it) for it in iterables)
    while active:
        it = active.popleft()
        try:
            item = next(it)
        except StopIteration:
            continue
        yield item
        active.append(it)

print(list(interleave("abc", [1], "xy")))

In [ ]:
import functools

def once(func):
    state = {}

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if "result" not in state:
            state["result"] = func(*args, **kwargs)
        return state["result"]

    wrapper.reset = state.clear
    return wrapper

@once
def answer():
    print("computing")
    return 42

print(answer(), answer())

In [ ]:
p = [2, 2, 2]
q = [(0, 1), (2, 3), (4, 5)]
r = 2
s = 1
t = False

print(p, q, r, s, t)

In [ ]:
from contextlib import contextmanager

class Timings:
    def __init__(self, clock):
        self.clock = clock
        self.totals = {}
        self._stack = []

    @contextmanager
    def section(self, name):
        self._stack.append(name)
        path = "/".join(self._stack)
        start = self.clock()
        try:
            yield
        finally:
            self.totals[path] = self.totals.get(path, 0) + self.clock() - start
            self._stack.pop()

    def report(self):
        return sorted(self.totals.items())

now = [0]
t = Timings(lambda: now[0])
with t.section("outer"):
    now[0] = 2
    with t.section("inner"):
        now[0] = 5
print(t.report())

In [ ]:
def aliases(obj):
    seen = set()
    repeated = 0
    stack = [obj]
    while stack:
        item = stack.pop()
        if not isinstance(item, (list, tuple, dict)):
            continue
        if id(item) in seen:
            repeated += 1
            continue
        seen.add(id(item))
        stack.extend(item.values() if isinstance(item, dict) else item)
    return len(seen), repeated

shared = [1]
print(aliases([shared, shared, {"k": shared}]))

In [ ]:
class Settings:
    def __init__(self, parent=None, **values):
        object.__setattr__(self, "_parent", parent)
        object.__setattr__(self, "_values", dict(values))

    def __getattr__(self, name):
        if name.startswith("_"):
            raise AttributeError(name)
        if name in self._values:
            return self._values[name]
        if self._parent is not None:
            return getattr(self._parent, name)
        raise AttributeError(f"no setting {name!r}")

    def __setattr__(self, name, value):
        raise AttributeError("settings are read-only")

    def __delattr__(self, name):
        raise AttributeError("settings are read-only")

    def override(self, **values):
        return Settings(self, **values)

    def to_dict(self):
        merged = self._parent.to_dict() if self._parent is not None else {}
        merged.update(self._values)
        return merged

base = Settings(colour="blue")
print(base.override(size=3).to_dict())

In [ ]:
import sys
import importlib.abc
import importlib.util

class SourceFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def __init__(self, sources):
        self.sources = dict(sources)
        self.loaded = []

    def find_spec(self, name, path=None, target=None):
        if name in self.sources:
            return importlib.util.spec_from_loader(name, self)
        return None

    def exec_module(self, module):
        self.loaded.append(module.__name__)
        exec(self.sources[module.__name__], module.__dict__)

def install(sources):
    finder = SourceFinder(sources)
    sys.meta_path.insert(0, finder)
    return finder

def uninstall(finder):
    if finder in sys.meta_path:
        sys.meta_path.remove(finder)
    for name in finder.loaded:
        sys.modules.pop(name, None)

finder = install({"tiny_mod": "GREETING = 'hello'"})
import tiny_mod
print(tiny_mod.GREETING, finder.loaded)
uninstall(finder)

In [ ]:
import heapq

def gather_limited(coros, limit):
    results = [None] * len(coros)
    queue, seq, next_index, total = [], 0, 0, 0

    def start(now):
        nonlocal seq, next_index
        heapq.heappush(queue, (now, seq, next_index))
        seq += 1
        next_index += 1

    while next_index < min(limit, len(coros)):
        start(0)
    while queue:
        now, _, i = heapq.heappop(queue)
        try:
            delay = next(coros[i])
        except StopIteration as stop:
            results[i] = stop.value
            total = max(total, now)
            if next_index < len(coros):
                start(now)
            continue
        heapq.heappush(queue, (now + delay, seq, i))
        seq += 1
    return results, total

def job(waits, result):
    for w in waits:
        yield w
    return result

print(gather_limited([job([3], "a"), job([1, 1], "b"), job([1], "c")], 2))

In [ ]:
import heapq
from contextlib import contextmanager

class Builder:
    def __init__(self):
        self.rules = {}
        self.cache = {}
        self._planned = None

    def rule(self, name, deps=(), cost=1):
        def decorate(func):
            if name in self.rules:
                raise ValueError(f"rule {name!r} already exists")
            self.rules[name] = (func, tuple(deps), cost)
            return func
        return decorate

    def order(self, target):
        result, state = [], {}

        def visit(name, path):
            if name not in self.rules:
                raise KeyError(name)
            if state.get(name) == "done":
                return
            if state.get(name) == "visiting":
                raise ValueError("cycle: " + " -> ".join(path + [name]))
            state[name] = "visiting"
            for dep in self.rules[name][1]:
                visit(dep, path + [name])
            state[name] = "done"
            result.append(name)

        visit(target, [])
        return result

    def build(self, target):
        names = self.order(target)
        if self._planned is not None:
            self._planned.extend(n for n in names if n not in self.cache)
            return None
        for name in names:
            if name not in self.cache:
                func, deps, _ = self.rules[name]
                self.cache[name] = func(*[self.cache[d] for d in deps])
        return self.cache[target]

    def invalidate(self, name):
        stale = {name}
        changed = True
        while changed:
            changed = False
            for rule, (_, deps, _) in self.rules.items():
                if rule not in stale and any(d in stale for d in deps):
                    stale.add(rule)
                    changed = True
        for rule in stale:
            self.cache.pop(rule, None)

    @contextmanager
    def dry_run(self):
        planned, previous = [], self._planned
        self._planned = planned
        try:
            yield planned
        finally:
            self._planned = previous

    def timeline(self, target, workers):
        names = self.order(target)
        rank = {n: i for i, n in enumerate(names)}
        waiting = {n: set(self.rules[n][1]) for n in names}
        ready = [rank[n] for n in names if not waiting[n]]
        heapq.heapify(ready)
        running, starts, now = [], {}, 0
        while ready or running:
            while ready and len(running) < workers:
                name = names[heapq.heappop(ready)]
                starts[name] = now
                heapq.heappush(running, (now + self.rules[name][2], rank[name], name))
            end, _, name = heapq.heappop(running)
            now, finished = end, [name]
            while running and running[0][0] == end:
                finished.append(heapq.heappop(running)[2])
            for done in finished:
                for n in names:
                    if done in waiting[n]:
                        waiting[n].discard(done)
                        if not waiting[n]:
                            heapq.heappush(ready, rank[n])
        return starts, now

b = Builder()

@b.rule("a", cost=2)
def a():
    return 1

@b.rule("b", deps=["a"])
def b_rule(x):
    return x + 1

print(b.order("b"), b.build("b"), b.timeline("b", 2))
with b.dry_run() as planned:
    b.invalidate("a")
    b.build("b")
print(planned)

### Exercise: Second chance: merging sorted streams (B1)

Write a generator `merge_sorted(*iterables)`: each iterable is sorted; yield all their items in sorted order, lazily (holding at most one item from each iterable at a time), so it works when some are infinite. Use `heapq`, but not `heapq.merge`. When items are equal, the one from the earlier iterable comes first.

In [ ]:
import heapq

def merge_sorted(*iterables):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_merge():\n    \"a lazy merge\"\n    import itertools\n    assert list(merge_sorted([1, 4, 9], [2, 3], [], [5])) == [1, 2, 3, 4, 5, 9]\n    assert list(itertools.islice(merge_sorted(itertools.count(0, 2), itertools.count(1, 2)), 5)) == [0, 1, 2, 3, 4]\n    assert list(merge_sorted()) == [] and \"heapq.merge\" not in source and \"merge(\" not in source.replace(\"merge_sorted(\", \"\")\n    pairs = list(merge_sorted([(1, \"a\")], [(1, \"a\")]))\n    assert pairs == [(1, \"a\"), (1, \"a\")]")

### Exercise: Second chance: at most n times (B2)

Write a decorator factory `at_most(n)`: the decorated function runs for its first `n` calls; after that, calls don't run it and return the result of the last call that did. The wrapper's `calls` attribute counts how many times the function really ran. Keep the function's name.

In [ ]:
import functools

def at_most(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_at_most():\n    \"a limited decorator\"\n    @at_most(2)\n    def double(x):\n        return x * 2\n    assert [double(1), double(2), double(3), double(4)] == [2, 4, 4, 4]\n    assert double.calls == 2 and double.__name__ == \"double\"\n    @at_most(0)\n    def never():\n        return 1\n    assert never() is None and never.calls == 0")

### Exercise: Second chance: a temporary attribute (B4)

Write a context manager `patched(obj, name, value)` (with `@contextmanager`) that sets `obj.name` to `value` for the block, then puts back the old value, or deletes the attribute if it didn't exist before, even when the block raises. It yields `value`, and nested uses on the same attribute must unwind correctly.

In [ ]:
from contextlib import contextmanager

@contextmanager
def patched(obj, name, value):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_patched():\n    \"set-up and clean-up\"\n    class Box:\n        size = 1\n    b = Box()\n    b.colour = \"red\"\n    with patched(b, \"colour\", \"blue\") as v:\n        assert v == \"blue\" and b.colour == \"blue\"\n        with patched(b, \"colour\", \"green\"):\n            assert b.colour == \"green\"\n        assert b.colour == \"blue\"\n    assert b.colour == \"red\"\n    try:\n        with patched(b, \"label\", \"x\"):\n            raise ValueError\n    except ValueError:\n        pass\n    assert not hasattr(b, \"label\")\n    with patched(b, \"size\", 5):\n        assert b.size == 5\n    assert b.size == 1 and \"size\" not in vars(b)")

### Exercise: Second chance: counting the leaves (B5)

Write `leaves(obj)`: the number of non-container values reachable from `obj` through lists, tuples and dict values, where a container reached more than once is only counted (and walked) the first time. Cycles must work. A non-container `obj` counts as one leaf.

In [ ]:
def leaves(obj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_leaves():\n    \"walking by identity\"\n    shared = [1, 2]\n    assert leaves([shared, shared, (3,)]) == 3 and leaves({\"a\": [1, [2, 3]], \"b\": \"x\"}) == 4\n    loop = [1]\n    loop.append(loop)\n    assert leaves(loop) == 1 and leaves(7) == 1 and leaves([]) == 0")

### Exercise: Second chance: a frozen record (B6)

Write a class `Frozen(**fields)` whose fields are readable as attributes but can't be changed: assigning or deleting any attribute raises `AttributeError("frozen")`. `replace(**changes)` returns a new `Frozen` with some fields changed (an unknown field raises `TypeError`). Two records are equal when their fields are, they can be used in sets (hash by the fields), and `repr` gives `Frozen(a=1, b='x')` with the fields in the order given.

In [ ]:
class Frozen:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_frozen():\n    \"immutability with __setattr__\"\n    r = Frozen(a=1, b=\"x\")\n    assert (r.a, r.b) == (1, \"x\") and repr(r) == \"Frozen(a=1, b='x')\"\n    for action in (lambda: setattr(r, \"a\", 2), lambda: delattr(r, \"a\"), lambda: setattr(r, \"c\", 3)):\n        try:\n            action()\n            assert False\n        except AttributeError as e:\n            assert str(e) == \"frozen\"\n    s = r.replace(b=\"y\")\n    assert s.b == \"y\" and r.b == \"x\" and r == Frozen(a=1, b=\"x\") and r != s\n    assert len({r, Frozen(a=1, b=\"x\"), s}) == 2\n    try:\n        r.replace(z=1)\n        assert False\n    except TypeError:\n        pass")

### Exercise: Second chance: the first to finish (B8)

Coroutines are generators yielding sleep times, as in B8. Write `race(coros)`: start them all at time 0 in list order, and run them until the **first** one finishes (ties: the one whose last step ran first). Return `(index, result, time)` for the winner, and close every other generator (call `.close()`, so their `finally` blocks run). An empty list raises `ValueError`.

In [ ]:
import heapq

def race(coros):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_race():\n    \"the first result, then cancel the rest\"\n    closed = []\n    def job(name, waits):\n        try:\n            for w in waits:\n                yield w\n            return name\n        finally:\n            closed.append(name)\n    assert race([job(\"slow\", [5]), job(\"fast\", [1, 1]), job(\"mid\", [3])]) == (1, \"fast\", 2)\n    assert sorted(closed) == [\"fast\", \"mid\", \"slow\"]\n    assert race([job(\"a\", [2]), job(\"b\", [2])]) == (0, \"a\", 2)\n    try:\n        race([])\n        assert False\n    except ValueError:\n        pass")

### Exercise: Second chance: build levels (C)

Write `levels(deps)`: `deps` maps each rule to a list of the rules it depends on (every dependency is itself a key). Return a list of levels: the first holds the rules with no dependencies, and each later level holds the rules whose dependencies are all in earlier levels; sort each level's names. If some rules can never be placed (a cycle), raise `ValueError` naming them, sorted, as in `"cycle among: x, y"`.

In [ ]:
def levels(deps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_levels():\n    \"levels of a dependency graph\"\n    deps = {\"config\": [], \"lib\": [\"config\"], \"docs\": [\"config\"], \"app\": [\"lib\", \"config\"], \"all\": [\"app\", \"docs\"]}\n    assert levels(deps) == [[\"config\"], [\"docs\", \"lib\"], [\"app\"], [\"all\"]]\n    assert levels({}) == [] and levels({\"a\": [], \"b\": []}) == [[\"a\", \"b\"]]\n    try:\n        levels({\"x\": [\"y\"], \"y\": [\"x\"], \"z\": []})\n        assert False\n    except ValueError as e:\n        assert str(e) == \"cycle among: x, y\"")

## Your score

Write down your total before reading on. Then, as you go through each problem, note **why** you lost any marks:

1. **Didn't know it**: a gap in what you've learned. Revise the topic.
2. **Knew it, but couldn't use it**: you'd recognise the idea in a lesson but couldn't write it from scratch. Redo that week's exercises without looking.
3. **Edge case**: the main case worked, but an empty input, an error in the middle of a block or a second run failed.
4. **Ran out of time**: the plan, not the knowledge, was the problem.

| Score | What it means |
|---|---|
| 144 or more (80%) | Distinction. You're well prepared for Phase 6. |
| 108 to 143 (60%) | Pass. Revise the topics where you lost marks during the rest week. |
| under 108 | Not yet. Spend the rest week on the topics below, then take the exam again. What matters is the second attempt. |

## Part A

| Questions | The idea | Revise |
|---|---|---|
| 1 | iterables give fresh iterators | [Week 59, Day 1](w059d1.html) |
| 2 | generators share their position | [Week 59, Day 2](w059d2.html) |
| 3 | groupby groups consecutive keys | [Week 59, Day 5](w059d5.html) |
| 4 | late binding in closures | [Week 60, Day 1](w060d1.html) |
| 5 | functools.wraps | [Week 60, Day 2](w060d2.html) |
| 6 | partial | [Week 60, Day 4](w060d4.html) |
| 7 | exceptions arrive at the yield | [Week 61, Day 3](w061d3.html) |
| 8 | ExitStack | [Week 61, Day 4](w061d4.html) |
| 9 | + makes a new list, += mutates | [Week 62, Day 2](w062d2.html) |
| 10 | weak references | [Week 62, Day 5](w062d5.html) |
| 11, 12 | lookup order and data descriptors | [Week 63, Days 1 and 2](w063d1.html) |
| 13, 14 | __init_subclass__ and metaclasses | [Week 63, Days 4 and 5](w063d4.html) |
| 15 | bytecode and code objects | [Week 64, Days 1 and 2](w064d1.html) |
| 16 | finders return specs | [Week 64, Day 5](w064d5.html) |
| 17 | the GIL and waiting | [Week 66, Day 3](w066d3.html) |
| 18 | races and locks | [Week 66, Day 4](w066d4.html) |
| 19 | blocking the event loop | [Week 67, Day 4](w067d4.html) |
| 20 | when to extend | [Week 68, Day 5](w068d5.html) |

Question 2 catches people who expect `zip(c, range(3))` to start counting again from 1: the generator remembers where it was, so it continues from 3. Question 12 is subtle: a property with no setter is still a data descriptor, which is why assigning to it raises an error instead of quietly creating an instance attribute.

## Part B

Each model solution runs, so try it. The notes give the key decision and the most common mistakes.

### B1: Taking turns

**The key decision**: keep the **iterators** (not the iterables) in a queue, and put one back only after it has produced an item, so used-up ones drop out. **Common mistakes**: `zip` or `zip_longest` (they stop early, or pad with fill values that then need filtering, and `zip` pulls an item from every iterable before yielding any); calling `iter()` again on every round, which restarts lists from the beginning; and pulling the next item before it's needed, which the laziness check catches.

### B2: Only once

**The key decision**: record "has a result" separately from the result itself. A test like `if result is None` runs the function again when it legitimately returns `None`. Assigning the result only **after** `func` returns means an exception leaves nothing remembered. **Common mistakes**: a module-level variable (shared by every decorated function); forgetting `functools.wraps`; and a `reset` that rebinds a local variable instead of changing the shared state (here, `state.clear` is the bound method of the shared dict, so it works with no `nonlocal`).

### B3: Predict

`p`: all three lambdas read `i` when they're called, after the loop has finished. `q`: `zip` takes two items **from the same iterator** for each tuple. `r`: `a[:]` is a copy. `s`: deleting the instance attribute uncovers the class attribute. `t`: calling `asyncio.sleep(1)` without `await` only creates a coroutine object (and Python warns that it was never awaited).

### B4: Nested timers

**The key decision**: a stack of the open sections' names gives the path, and `try`/`finally` around the `yield` records the time and pops the stack whatever happens in the block. **Common mistakes**: no `finally`, so the failing section was never recorded and every later path was wrong because its name stayed on the stack; storing only the latest time instead of adding repeats; and building the path after popping.

### B5: Counting references

**The key decision**: track containers by `id`, because equal containers (`[1]` and `[1]`) are still different objects, and comparing containers that hold cycles can recurse forever. Count a container as repeated **before** walking into it, and don't walk into it again. **Common mistakes**: a `seen` list compared with `in` (which uses `==`, so equal but distinct lists look the same, and comparing cyclic lists can hit the recursion limit); walking dict keys as well as values; and recursion that never checks for repeats, so the cycle loops forever.

### B6: Layered settings

**The key decision**: `__getattr__` only runs when normal lookup fails, so it's the right place for the chain, while `__setattr__` and `__delattr__` block changes. The constructor has to get round its own `__setattr__`, which is what `object.__setattr__` is for. **Common mistakes**: `self._values = ...` in `__init__`, which raises the read-only error; `__getattribute__` instead of `__getattr__` (it runs for `_values` too and recurses); and forgetting the `_`-name guard, so `copy.copy` (which looks up special names on a half-built object) recursed forever.

### B7: Modules from strings

**The key decision**: one object can be both the finder (`find_spec`, returning `None` for names it doesn't know, so other imports carry on as normal) and the loader (`exec_module`, running the source in the new module's namespace). **Common mistakes**: appending to `sys.meta_path` instead of inserting at the front; raising instead of returning `None` for unknown names, which breaks every other import; recording the module in `find_spec` (which can be called without loading); and forgetting to remove the modules from `sys.modules`, so they could still be imported after `uninstall`.

### B8: Gathering with a limit

**The key decision**: a heap of `(wake time, sequence number, index)` is the whole event loop; the sequence number settles ties in the order tasks went to sleep and stops the heap ever comparing generators. A finished task frees a slot by starting the next one **at its finish time**. **Common mistakes**: starting the next coroutine at time 0 or at the end of the round; returning results in finishing order instead of the list's order; and running the newly started coroutine immediately instead of through the heap, which breaks the tie order.

## Part C

**Where the marks went.**

- **order**: a depth-first search with **three** states. With only "seen", a rule needed by two others looks like a cycle; with only "done", a cycle recurses forever. Passing the path down gives the message.
- **build**: call each rule with its dependencies' cached results, `func(*[self.cache[d] for d in deps])`, and skip rules that are already cached.
- **invalidate**: dependents are found through the **reverse** of the `deps` relation; repeating until nothing changes catches indirect ones.
- **dry_run**: a flag (here, the list itself) that `build` checks, restored in `finally`. Saving and restoring the previous value lets dry runs nest.
- **timeline**: an event simulation with two heaps, one of ready rules by rank and one of running rules by end time. The usual errors were starting a rule before **all** its dependencies had finished, and not freeing every worker that finishes at the same moment before choosing what to start next.

## Second chances

Each of these practises the same skill as an exam problem on a fresh problem. Do the ones for the problems where you lost marks first.

## What's next

**Your revision plan.** For each problem where you lost marks, write one line: the problem, the kind of mistake, and the day you'll revisit. Do those days during the rest week. If you're retaking the exam, do it after the rest week, without looking at this page again first.

**Phase 6: engineering in practice.** After a week's rest, Phase 6 turns from the language to the work around it: testing in depth (fixtures, mocking, property-based and mutation testing), packaging, code quality and collaboration, databases, networking and HTTP, web backends, automation, security, system design, and Python for data and machine learning. Almost everything in it leans on this phase: fixtures are generators and context managers, mocks are attribute magic, web frameworks are built on decorators and `asyncio`, and many database drivers are C extensions.

You've finished Phase 5. Eleven weeks ago a `for` loop was just a loop; since then you've taken it apart into iterators and generators, wrapped functions in closures and decorators, made clean-up impossible to forget, followed every reference, intercepted attribute lookup and class creation, read the bytecode, timed and cached and vectorised, run work at once in threads, processes and event loops, called C from Python, and built an interpreter of your own. Take the rest week: you've earned it.